In [1]:
# Import necessary libraries
import pandas as pd
from datasets import load_dataset
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.multioutput import MultiOutputClassifier


In [2]:
# Loading faers-ascii-2024Q4 dataset

# Calling 'demographic' data set
file_path = '/Users/agnesnamyalo/Documents/NOTES/NOTES_SEM_2/NLP/PROJECT/ASCII/DEMO24Q4.txt'

# Read the $-delimited file
df_demo = pd.read_csv(file_path, delimiter='$')

/var/folders/ly/fvhqh8l970n41tr2rwsg6l8h0000gn/T/ipykernel_13062/1567419953.py:7: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_demo = pd.read_csv(file_path, delimiter='$')


In [3]:
df_demo.head()

,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,...,age_grp,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country
0,100100247,10010024,7,F,20080501.0,20241104,20140313,20241109,EXP,NaN,...,NaN,M,Y,NaN,NaN,20241109,NaN,MD,DE,DE
1,100373859,10037385,9,F,20120101.0,20241030,20140326,20241111,EXP,NaN,...,E,F,Y,64.3,KG,20241111,NaN,MD,GB,GB
2,1016611044,10166110,44,F,20140430.0,20241129,20140512,20241204,EXP,NaN,...,A,F,Y,NaN,NaN,20241204,NaN,MD,CA,CA
3,101735213,10173521,3,F,20140507.0,20241121,20140515,20241217,EXP,NaN,...,T,M,Y,68.0,KG,20241217,NaN,MD,GB,GB
4,1020101730,10201017,30,F,20140428.0,20241112,20140528,20241114,EXP,NaN,...,NaN,M,Y,NaN,NaN,20241114,NaN,HP,CA,CA


In [4]:
df_demo.columns

Index(['primaryid', 'caseid', 'caseversion', 'i_f_code', 'event_dt', 'mfr_dt',
       'init_fda_dt', 'fda_dt', 'rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr',
       'lit_ref', 'age', 'age_cod', 'age_grp', 'sex', 'e_sub', 'wt', 'wt_cod',
       'rept_dt', 'to_mfr', 'occp_cod', 'reporter_country', 'occr_country'],
      dtype='object')

In [5]:
# Keep only the selected columns
df_demo_small = df_demo[['primaryid', 'age', 'age_grp', 'sex', 'wt']]

In [6]:
df_demo_small.head()

,primaryid,age,age_grp,sex,wt
0,100100247,56.0,NaN,M,NaN
1,100373859,72.0,E,F,64.3
2,1016611044,34.0,A,F,NaN
3,101735213,13.0,T,M,68.0
4,1020101730,74.0,NaN,M,NaN


In [7]:
df_demo_small.shape

(410849, 5)

In [8]:
df_demo_small= df_demo_small.dropna(subset=["age", "sex"])  # Drop rows with missing critical data

In [9]:
df_demo_small.shape

(239844, 5)

In [10]:
# Count the number of missing values in the 'wt' column
missing_wt_count = df_demo_small['wt'].isnull().sum()

In [11]:
print(f"Number of missing values in 'wt': {missing_wt_count}")

Number of missing values in 'wt': 181646


In [12]:
# Filling missing values of 'wt' with median
df_demo_small["wt"] = df_demo_small["wt"].fillna(df_demo_small["wt"].median())

In [13]:
# Count the number of missing values in the 'wt' column
missing_wt_count = df_demo_small['wt'].isnull().sum()

print(f"Number of missing values in 'wt': {missing_wt_count}")

Number of missing values in 'wt': 0


In [14]:
# Get unique categories in the 'sex' column
unique_sex_categories = df_demo_small['sex'].unique()

print(f"Unique categories in 'sex': {unique_sex_categories}")

Unique categories in 'sex': ['M' 'F' 'UNK']


In [15]:
# Categorical into numeric

# Sex: Male=0, Female=1, Unknown=2
df_demo_small["sex"] = df_demo_small["sex"].map({"M": 0, "F": 1,"UNK":2}).fillna(2).astype(int)

In [16]:
df_demo_small.shape

(239844, 5)

In [17]:
# Check for duplicates of 'primaryid'
print(df_demo_small['primaryid'].duplicated().sum())

0


In [18]:
df_demo_small.head()

,primaryid,age,age_grp,sex,wt
0,100100247,56.0,NaN,0,71.0
1,100373859,72.0,E,1,64.3
2,1016611044,34.0,A,1,71.0
3,101735213,13.0,T,0,68.0
4,1020101730,74.0,NaN,0,71.0


In [19]:
# Calling 'drug' dataset
file_path = '/Users/agnesnamyalo/Documents/NOTES/NOTES_SEM_2/NLP/PROJECT/ASCII/DRUG24Q4.txt'  # <-- update this path

# Read the $-delimited file
df_drug = pd.read_csv(file_path, delimiter='$')

/var/folders/ly/fvhqh8l970n41tr2rwsg6l8h0000gn/T/ipykernel_13062/868165921.py:5: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_drug = pd.read_csv(file_path, delimiter='$')


In [20]:
df_drug.head()

,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,val_vbm,route,dose_vbm,cum_dose_chr,cum_dose_unit,dechal,rechal,lot_num,exp_dt,nda_num,dose_amt,dose_unit,dose_form,dose_freq
0,100100247,10010024,1,PS,ADALIMUMAB,ADALIMUMAB,1,Subcutaneous,NaN,NaN,NaN,Y,NaN,Not Available,NaN,125057.0,40.0,MG,NaN,QOW
1,100100247,10010024,2,SS,ADALIMUMAB,ADALIMUMAB,1,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN,125057.0,NaN,NaN,NaN,NaN
2,100100247,10010024,3,SS,CYCLOSPORINE,CYCLOSPORINE,1,Unknown,NaN,NaN,NaN,D,NaN,Not Available,NaN,65003.0,NaN,NaN,NaN,NaN
3,100100247,10010024,4,SS,REMICADE,INFLIXIMAB,1,Intravenous (not otherwise specified),6 infusions at 8 weeks interval,NaN,NaN,Y,NaN,Not Available,NaN,NaN,NaN,NaN,NaN,NaN
4,100100247,10010024,5,SS,ETANERCEPT,ETANERCEPT,1,Subcutaneous,NaN,NaN,NaN,Y,NaN,Not Available,NaN,NaN,100.0,MG,NaN,/WK


In [21]:
df_drug.columns

Index(['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai',
       'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit',
       'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt',
       'dose_unit', 'dose_form', 'dose_freq'],
      dtype='object')

In [22]:
# Keep only the selected columns
df_drug_small = df_drug[['primaryid','drug_seq', 'drugname', 'prod_ai','route','dose_amt','dose_unit']]

In [23]:
df_drug_small.head()

,primaryid,drug_seq,drugname,prod_ai,route,dose_amt,dose_unit
0,100100247,1,ADALIMUMAB,ADALIMUMAB,Subcutaneous,40.0,MG
1,100100247,2,ADALIMUMAB,ADALIMUMAB,NaN,NaN,NaN
2,100100247,3,CYCLOSPORINE,CYCLOSPORINE,Unknown,NaN,NaN
3,100100247,4,REMICADE,INFLIXIMAB,Intravenous (not otherwise specified),NaN,NaN
4,100100247,5,ETANERCEPT,ETANERCEPT,Subcutaneous,100.0,MG


In [24]:
# Available routes
df_drug_small['route'].unique()

array(['Subcutaneous', nan, 'Unknown',
       'Intravenous (not otherwise specified)', 'Topical', 'Oral',
       'Other', 'Intravenous drip', 'Intramuscular', 'Ophthalmic',
       'Respiratory (inhalation)', 'Parenteral', 'Intravenous bolus',
       'Intravenous use', 'Subcutaneous use', 'Oral use', 'Sublingual',
       'Transdermal', 'Transplacental', 'Nasal', 'Intra-arterial',
       'Intravesical', 'Intracardiac', 'Buccal', 'Intra-uterine',
       'Vaginal', 'Urethral', 'Oropharingeal', 'Periarticular',
       'Endocervical', 'Intrameningeal', 'Intra-articular',
       'Sunconjunctival', 'Intraocular', 'Rectal', 'Intrathecal',
       'Cutaneous', 'Epidural', 'Vaginal use', 'Intramuscular use',
       'Rectal use', 'Intracavernous', 'Intracervical',
       'Occlusive dressing technique', 'Intralesional', 'Inhalation use',
       'Intratracheal', 'Dental', 'Endosinusial', 'Auricular (otic)',
       'Iontophoresis', 'Nasal use', 'Ocular use', 'Intradermal',
       'Intracardiac use', '

In [25]:
# Create a route mapping dictionary for standardization
route_mapping = {
    # Parent routes
    r'(?i)subcutaneous': 'Subcutaneous',
    r'(?i)intravenous': 'Intravenous',
    r'(?i)oral': 'Oral',
    r'(?i)topical': 'Topical',
    r'(?i)ophthalmic': 'Ophthalmic',
    r'(?i)respiratory': 'Respiratory',
    r'(?i)intramuscular': 'Intramuscular',
    r'(?i)rectal': 'Rectal',
    r'(?i)vaginal': 'Vaginal',
    r'(?i)transdermal': 'Transdermal',
    r'(?i)nasal': 'Nasal',
    r'(?i)buccal': 'Buccal',
    r'(?i)intrathecal': 'Intrathecal',
    r'(?i)cutaneous': 'Cutaneous',
    r'(?i)epidural': 'Epidural',

    # Special cases
    r'(?i)unknown': 'Unknown',
    r'(?i)other': 'Other',
    r'(?i)parenteral': 'Parenteral',
    r'(?i)iontophoresis': 'Iontophoresis',
    r'(?i)hemodialysis': 'Hemodialysis',

    # Group less common routes
    r'(?i)intra-arterial|intraarterial': 'Intra-arterial',
    r'(?i)intravesical|intracavitary': 'Intravesical',
    r'(?i)intracardiac|intracoronary': 'Intracardiac',
    r'(?i)intra-uterine|intrauterine': 'Intrauterine',
    r'(?i)intraocular|retrobulbar': 'Intraocular',
    r'(?i)intraperitoneal|intrapericardial': 'Intraperitoneal',
    r'(?i)intradermal|intralesional': 'Intradermal',

    # Catch-all for rare routes
    r'(?i)use$': '',  # Remove trailing "use"
    r'(?i)not otherwise specified': '',  # Remove NOS
}

def clean_route(route):
    """Clean and standardize route values."""
    if pd.isna(route):
        return 'Unknown'

    # Clean common patterns first
    route = str(route).lower()
    route = route.replace('(not otherwise specified)', '').strip()
    route = route.replace(' use', '').strip()

    # Apply mapping
    for pattern, replacement in route_mapping.items():
        if pd.notna(route) and re.search(pattern, route):
            return replacement

    # Final cleanup for remaining values
    route = route.title()  # Standardize capitalization
    return 'Other' if route not in route_mapping.values() else route

# Apply cleaning
df_drug_small['route_clean'] = df_drug_small['route'].apply(clean_route)

# Handle remaining NaNs and unknowns
df_drug_small['route_clean'] = df_drug_small['route_clean'].replace({
    np.nan: 'Unknown',
    '': 'Unknown',
    'Unknown': 'Unknown'
})

# Verify results
print("Cleaned Route Distribution:")
print(df_drug_small['route_clean'].value_counts(dropna=False))

Cleaned Route Distribution:
route_clean
Unknown            1216510
Oral                316878
Subcutaneous        205002
Intravenous         196563
Other                38585
Topical               9404
Intramuscular         8481
Respiratory           6362
Intracardiac          5888
Ophthalmic            4949
Nasal                 2896
Cutaneous             2542
Transdermal           2406
Intrauterine          2202
Intra-arterial        1970
Vaginal               1556
Buccal                1518
Intrathecal           1514
Intraperitoneal       1506
Rectal                1464
Intraocular           1117
Parenteral             626
Intravesical           584
Intradermal            249
Epidural               122
Iontophoresis           24
Hemodialysis            20
Name: count, dtype: int64


/var/folders/ly/fvhqh8l970n41tr2rwsg6l8h0000gn/T/ipykernel_13062/4290422084.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drug_small['route_clean'] = df_drug_small['route'].apply(clean_route)
/var/folders/ly/fvhqh8l970n41tr2rwsg6l8h0000gn/T/ipykernel_13062/4290422084.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drug_small['route_clean'] = df_drug_small['route_clean'].replace({


In [26]:
df_drug_small.head()

,primaryid,drug_seq,drugname,prod_ai,route,dose_amt,dose_unit,route_clean
0,100100247,1,ADALIMUMAB,ADALIMUMAB,Subcutaneous,40.0,MG,Subcutaneous
1,100100247,2,ADALIMUMAB,ADALIMUMAB,NaN,NaN,NaN,Unknown
2,100100247,3,CYCLOSPORINE,CYCLOSPORINE,Unknown,NaN,NaN,Unknown
3,100100247,4,REMICADE,INFLIXIMAB,Intravenous (not otherwise specified),NaN,NaN,Intravenous
4,100100247,5,ETANERCEPT,ETANERCEPT,Subcutaneous,100.0,MG,Subcutaneous


In [27]:
# Drop rows with missing critical fields (e.g., drugname, route)
grouped_df= df_drug_small.dropna(subset=["drugname", "route_clean"])

In [28]:
# Calling reaction data set
file_path = '/Users/agnesnamyalo/Documents/NOTES/NOTES_SEM_2/NLP/PROJECT/ASCII/REAC24Q4.txt'  # <-- update this path

# Read the $-delimited file
df_reac = pd.read_csv(file_path, delimiter='$')

In [29]:
df_reac.head()

,primaryid,caseid,pt,drug_rec_act
0,100100247,10010024,Malignant melanoma stage I,NaN
1,100100247,10010024,Keratoacanthoma,NaN
2,100100247,10010024,Blood pressure increased,NaN
3,100100247,10010024,Keratoacanthoma,NaN
4,100100247,10010024,Hyperkeratosis,NaN


In [30]:
df_reac.columns

Index(['primaryid', 'caseid', 'pt', 'drug_rec_act'], dtype='object')

In [31]:
# Keep only the selected columns
df_reac_small = df_reac[['primaryid','pt']]

In [32]:
df_reac_small.head()

,primaryid,pt
0,100100247,Malignant melanoma stage I
1,100100247,Keratoacanthoma
2,100100247,Blood pressure increased
3,100100247,Keratoacanthoma
4,100100247,Hyperkeratosis


In [33]:
df_reac_small.shape

(1472750, 2)

In [34]:
df_reac_small.shape

(1472750, 2)

In [35]:
# Drop duplicate rows based on primaryid and pt
df_reac_small = df_reac_small.drop_duplicates(subset=['primaryid', 'pt'])

In [36]:
df_reac_small.shape

(1452498, 2)

## Started joining dataframes

In [37]:
merged = pd.merge(
    pd.merge(
        df_demo_small[["primaryid", "age", "sex", "wt"]],
        grouped_df[["primaryid", "drugname", "route_clean", "dose_amt"]],
        on="primaryid",  # Correct key
        how="inner"
    ),
    df_reac_small[["primaryid", "pt"]],
    on="primaryid",  # Correct key
    how="inner"
)


In [38]:
merged.head()

,primaryid,age,sex,wt,drugname,route_clean,dose_amt,pt
0,100100247,56.0,0,71.0,ADALIMUMAB,Subcutaneous,40.0,Malignant melanoma stage I
1,100100247,56.0,0,71.0,ADALIMUMAB,Subcutaneous,40.0,Keratoacanthoma
2,100100247,56.0,0,71.0,ADALIMUMAB,Subcutaneous,40.0,Blood pressure increased
3,100100247,56.0,0,71.0,ADALIMUMAB,Subcutaneous,40.0,Hyperkeratosis
4,100100247,56.0,0,71.0,ADALIMUMAB,Subcutaneous,40.0,Therapy non-responder


In [216]:
# Save to CSV
merged.to_csv("faers-ascii-2024Q4_4.csv", index=False)


Modeling


In [217]:
# Load merged dataset
df = pd.read_csv("scratch/faers-ascii-2024Q4_4.csv")

In [218]:
# Preview data
print(df.head())

   primaryid   age  sex    wt    drugname   route_clean  dose_amt  \
0  100100247  56.0    0  71.0  ADALIMUMAB  Subcutaneous      40.0   
1  100100247  56.0    0  71.0  ADALIMUMAB  Subcutaneous      40.0   
2  100100247  56.0    0  71.0  ADALIMUMAB  Subcutaneous      40.0   
3  100100247  56.0    0  71.0  ADALIMUMAB  Subcutaneous      40.0   
4  100100247  56.0    0  71.0  ADALIMUMAB  Subcutaneous      40.0   

                           pt  
0  Malignant melanoma stage I  
1             Keratoacanthoma  
2    Blood pressure increased  
3              Hyperkeratosis  
4       Therapy non-responder  


In [219]:
df.columns

Index(['primaryid', 'age', 'sex', 'wt', 'drugname', 'route_clean', 'dose_amt',
       'pt'],
      dtype='object')

In [220]:
df = df.groupby(
    ['primaryid', 'age', 'sex', 'wt', 'drugname', 'route_clean', 'dose_amt']
)['pt'].agg(list).reset_index()

In [221]:
df.head()

,primaryid,age,sex,wt,drugname,route_clean,dose_amt,pt
0,39703653,59.0,0,88.7,CYCLOPHOSPHAMIDE,Unknown,100.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
1,39703653,59.0,0,88.7,CYCLOSPORINE,Oral,3.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
2,39703653,59.0,0,88.7,ENALAPRIL,Oral,20.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
3,39703653,59.0,0,88.7,ENALAPRIL,Unknown,10.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
4,39703653,59.0,0,88.7,PREDNISONE,Unknown,24.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."


In [222]:
df.shape

(434301, 8)

In [223]:
# Fill missing values for numerical columns with 'median'
df['age'] = df['age'].fillna(df['age'].median())
df['wt'] = df['wt'].fillna(df['wt'].median())

In [224]:
df.shape

(434301, 8)

In [225]:
# Fill missing values for categorical columns with 'unknown'
df['sex'] = df['sex'].fillna('Unknown')
df['route_clean'] = df['route_clean'].fillna('Unknown')

In [226]:
df.shape

(434301, 8)

In [227]:
# Drop rows with missing drug names and adverse effects
df = df.dropna(subset=['drugname', 'pt'])

In [228]:
df.shape

(434301, 8)

In [229]:
df.head()

,primaryid,age,sex,wt,drugname,route_clean,dose_amt,pt
0,39703653,59.0,0,88.7,CYCLOPHOSPHAMIDE,Unknown,100.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
1,39703653,59.0,0,88.7,CYCLOSPORINE,Oral,3.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
2,39703653,59.0,0,88.7,ENALAPRIL,Oral,20.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
3,39703653,59.0,0,88.7,ENALAPRIL,Unknown,10.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
4,39703653,59.0,0,88.7,PREDNISONE,Unknown,24.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."


In [230]:
#Clean and encode categorical variables

# Tokenize drug names for text modeling
#df['drugname'] = df['drugname'].str.split(', ')#splitting the drug names into individual words

In [231]:
# One-hot encode routes
#the encoder to ignore any unknown categories during the transformation process
encoder = OneHotEncoder(handle_unknown='ignore')
encoded_routes = encoder.fit_transform(df[['route_clean']]).toarray()
route_columns = encoder.get_feature_names_out(['route_clean'])

In [232]:
df.shape

(434301, 8)

In [233]:
# Multi-hot encode adverse effects
mlb = MultiLabelBinarizer()
encoded_pt = mlb.fit_transform(df['pt'])
pt_columns = mlb.classes_

In [234]:
df.shape

(434301, 8)

In [235]:
df.head()

,primaryid,age,sex,wt,drugname,route_clean,dose_amt,pt
0,39703653,59.0,0,88.7,CYCLOPHOSPHAMIDE,Unknown,100.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
1,39703653,59.0,0,88.7,CYCLOSPORINE,Oral,3.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
2,39703653,59.0,0,88.7,ENALAPRIL,Oral,20.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
3,39703653,59.0,0,88.7,ENALAPRIL,Unknown,10.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."
4,39703653,59.0,0,88.7,PREDNISONE,Unknown,24.0,"[Nephropathy toxic, Pruritus, Hypertension, Oc..."


In [236]:
# Combine features into a single DataFrame
structured_data = pd.concat([
    df[['age', 'wt']],  # Numerical features
    pd.DataFrame(encoded_routes, columns=route_columns),  # Encoded routes
], axis=1)

print("Structured Data Shape:", structured_data.shape)

Structured Data Shape: (434301, 29)


In [237]:
structured_data.shape

(434301, 29)

In [238]:
structured_data.head()

,age,wt,route_clean_Buccal,route_clean_Cutaneous,route_clean_Epidural,route_clean_Hemodialysis,route_clean_Intra-arterial,route_clean_Intracardiac,route_clean_Intradermal,route_clean_Intramuscular,...,route_clean_Oral,route_clean_Other,route_clean_Parenteral,route_clean_Rectal,route_clean_Respiratory,route_clean_Subcutaneous,route_clean_Topical,route_clean_Transdermal,route_clean_Unknown,route_clean_Vaginal
0,59.0,88.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,59.0,88.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,59.0,88.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,59.0,88.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,59.0,88.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [239]:
# Text normalization
df['drugname'] = df['drugname'].str.lower()
df['route_clean'] = df['route_clean'].str.lower()
df['pt'] = df['pt'].apply(lambda ade_list: [ade.lower() for ade in ade_list])

In [240]:
df.columns

Index(['primaryid', 'age', 'sex', 'wt', 'drugname', 'route_clean', 'dose_amt',
       'pt'],
      dtype='object')

In [241]:
# Features (structured + drug names)
X_structured = df[['age', 'wt', 'route_clean']]
X_text = df['drugname']  # Drug names for text modeling

In [242]:
# Labels (multi-hot encoded adverse effects)
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['pt'])

In [243]:
print(f"Features shape (structured): {X_structured.shape}")
print(f"Features shape (text): {len(X_text)}")
print(f"Labels shape: {y.shape}")

Features shape (structured): (434301, 3)
Features shape (text): 434301
Labels shape: (434301, 10158)


In [244]:
# Train/Test split

#sample_size = int(len(X_structured))
sample_size = 20000
X_structured_small = X_structured[:sample_size]
X_text_small = X_text[:sample_size]
y_small = y[:sample_size]

X_struct_train, X_struct_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_structured_small, X_text_small, y_small, test_size=0.2, random_state=42
)

print(f"Train set size: {X_struct_train.shape[0]}")
print(f"Test set size: {X_struct_test.shape[0]}")

Train set size: 16000
Test set size: 4000


In [245]:
#Scale Numerical Features
scaler = StandardScaler()

# Scale structured features
X_struct_train_scaled = scaler.fit_transform(X_struct_train[['age', 'wt']])
X_struct_test_scaled = scaler.transform(X_struct_test[['age', 'wt']])

In [246]:
# Combine scaled numerical features with categorical features
X_struct_train_final = pd.concat([
    pd.DataFrame(X_struct_train_scaled, columns=['age', 'wt']),
    pd.get_dummies(X_struct_train['route_clean'])
], axis=1)

X_struct_test_final = pd.concat([
    pd.DataFrame(X_struct_test_scaled, columns=['age', 'wt']),
    pd.get_dummies(X_struct_test['route_clean'])
], axis=1)

print(f"Final structured train shape: {X_struct_train_final.shape}")
print(f"Final structured test shape: {X_struct_test_final.shape}")

Final structured train shape: (19216, 25)
Final structured test shape: (7219, 22)


In [247]:
#Convert lists of drug names into single strings:
# Convert lists (from CSV parsing) to strings
X_text_train = X_text_train.apply(lambda x: str(x).split(", ")[0] if isinstance(x, list) else str(x))
X_text_test = X_text_test.apply(lambda x: str(x).split(", ")[0] if isinstance(x, list) else str(x))

In [248]:
# Fill NaN with "Unknown"
X_text_train = X_text_train.fillna("Unknown")
X_text_test = X_text_test.fillna("Unknown")

In [249]:
#Preparing the drug names for use with a BERT

tokenizer = BertTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

def tokenize_drugnames(drug_names):
    return tokenizer(
        drug_names.tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

In [250]:
# Tokenize(drugnames)
text_inputs_train = tokenize_drugnames(X_text_train)
text_inputs_test = tokenize_drugnames(X_text_test)

In [251]:
# Ensure all columns in X_struct_train and X_struct_test are numeric
# ensures that your structured data is in the correct numeric format and then converts it into PyTorch tensors, making it ready to be used as input to a PyTorch model.
X_struct_train = X_struct_train.apply(pd.to_numeric, errors='coerce').fillna(0)
X_struct_test = X_struct_test.apply(pd.to_numeric, errors='coerce').fillna(0)

# Convert structured data to PyTorch tensors
X_struct_train_tensor = torch.tensor(X_struct_train.values, dtype=torch.float32)
X_struct_test_tensor = torch.tensor(X_struct_test.values, dtype=torch.float32)

In [252]:
# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [253]:
# Define the MultimodalADEModel class
class MultimodalADEModel(nn.Module):
    def __init__(self, num_structured_features, num_classes):
        super().__init__()
        # Text feature extractor (BERT)
        self.text_model = BertModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.text_model.eval()  # Freeze BERT
        self.text_fc = nn.Sequential(
            nn.Linear(self.text_model.config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Structured feature extractor
        self.structured_fc = nn.Sequential(
            nn.Linear(num_structured_features, 128),
            nn.ReLU()
        )

        # Final classification layer
        self.classifier = nn.Linear(128 * 2, num_classes)

    def forward(self, input_ids, attention_mask, struct_input):
        # Process text features
        with torch.no_grad():  # Disable gradients for BERT if frozen
            text_output = self.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        text_features = text_output.last_hidden_state.mean(dim=1)  # Mean pooling
        text_features = self.text_fc(text_features)

        # Process structured features
        structured_features = self.structured_fc(struct_input)

        # Concatenate features
        combined_features = torch.cat((text_features, structured_features), dim=1)

        # Classification (return logits, not probabilities)
        return self.classifier(combined_features)



In [254]:
# Train the model
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model
model = MultimodalADEModel(
    num_structured_features=X_struct_train_tensor.shape[1],
    num_classes=y_train_tensor.shape[1]
).to(device)

# Optimizer and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

# DataLoader
dataset_train = TensorDataset(
    text_inputs_train['input_ids'], 
    text_inputs_train['attention_mask'],
    X_struct_train_tensor,
    y_train_tensor
)
dataloader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in dataloader_train:
        optimizer.zero_grad()

        # Move batch to device
        input_ids_batch = batch[0].to(device)
        attention_mask_batch = batch[1].to(device)
        struct_batch = batch[2].to(device)
        labels_batch = batch[3].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids_batch,
            attention_mask=attention_mask_batch,
            struct_input=struct_batch
        )

        # Loss
        loss = criterion(outputs, labels_batch)
        
        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader_train):.4f}")

Using device: cuda
Epoch 1/5, Loss: 0.1370
Epoch 2/5, Loss: 0.0070
Epoch 3/5, Loss: 0.0066
Epoch 4/5, Loss: 0.0065
Epoch 5/5, Loss: 0.0063


In [255]:
# Re-initialize model (if needed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalADEModel(
    num_structured_features=X_struct_train_tensor.shape[1],
    num_classes=y_train_tensor.shape[1]
).to(device)

# Load trained weights (if saved)
# model.load_state_dict(torch.load("model_weights.pth"))

# Move test data to device
X_struct_test_tensor = X_struct_test_tensor.to(device)
text_inputs_test = {k: v.to(device) for k, v in text_inputs_test.items()}

# Evaluate
model.eval()
with torch.no_grad():
    outputs_test = model(
        input_ids=text_inputs_test['input_ids'],
        attention_mask=text_inputs_test['attention_mask'],
        struct_input=X_struct_test_tensor
    )
    predictions_test = (torch.sigmoid(outputs_test) > 0.5).float().cpu().numpy()


In [256]:
# Calculate metrics (after ensuring y_test and predictions_test are defined)
accuracy_micro = accuracy_score(y_test.flatten(), predictions_test.flatten())
f1_micro = f1_score(y_test.flatten(), predictions_test.flatten(), average='micro')
f1_macro = f1_score(y_test.flatten(), predictions_test.flatten(), average='macro')

print(f"Accuracy (Micro): {accuracy_micro:.4f}")
print(f"F1 Score (Micro): {f1_micro:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")


Accuracy (Micro): 0.5036
F1 Score (Micro): 0.5036
F1 Score (Macro): 0.3360


In [257]:
# After fitting the OneHotEncoder during training:
route_encoder = encoder  # Your actual encoder variable name

# Get encoded route feature names (e.g., ['route_clean_Intravenous', 'route_clean_Oral', ...])
encoded_route_columns = route_encoder.get_feature_names_out(['route_clean'])

# Extract original route names by removing the prefix
original_routes = [col.split("_")[-1] for col in encoded_route_columns]

# Create mapping: route name → encoded index
route_mapping = {route: idx for idx, route in enumerate(original_routes)}

# Print available routes
print("Route Name → Encoded Index:")
for route, idx in route_mapping.items():
    print(f"- {route}: {idx}")


Route Name → Encoded Index:
- Buccal: 0
- Cutaneous: 1
- Epidural: 2
- Hemodialysis: 3
- Intra-arterial: 4
- Intracardiac: 5
- Intradermal: 6
- Intramuscular: 7
- Intraocular: 8
- Intraperitoneal: 9
- Intrathecal: 10
- Intrauterine: 11
- Intravenous: 12
- Intravesical: 13
- Iontophoresis: 14
- Nasal: 15
- Ophthalmic: 16
- Oral: 17
- Other: 18
- Parenteral: 19
- Rectal: 20
- Respiratory: 21
- Subcutaneous: 22
- Topical: 23
- Transdermal: 24
- Unknown: 25
- Vaginal: 26


In [258]:
# checking the model

def predict_ade(drug_name, age=45, route_encoded=22, sex_encoded=1):

    # Tokenize drug name
    text_inputs = tokenizer(
        [drug_name],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    # Prepare structured features
    struct_features = torch.tensor(
        [[age,route_encoded, sex_encoded]],
        dtype=torch.float32
    ).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=text_inputs['input_ids'],
            attention_mask=text_inputs['attention_mask'],
            struct_input=struct_features
        )
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    # Get top 5 predictions
    top_indices = np.argsort(probs)[-5:][::-1]
    return [(mlb.classes_[i], round(probs[i], 2)) for i in top_indices]

# Example usage
drug = "ADALIMUMAB"
predictions = predict_ade(drug)
print(f"Predicted Adverse Effects for {drug}:")
for ade, prob in predictions:
    print(f"- {ade}")
    

Predicted Adverse Effects for ADALIMUMAB:
- achlorhydria
- ankle deformity
- benign salivary gland neoplasm
- streptococcal infection
- klebsiella urinary tract infection
